# Activity 3: Implement Smart Governance Rules
**Blockchain Wonders workshop · about 30 minutes**

A ledger records what happened. It does not decide what is **allowed** to happen. That is the job of a **smart contract**: rules that run the same way for everyone and write what they did to the ledger.

Our contract turns agent recommendations into final decisions:

```
agents submit decisions  ->  contract checks the rules
validators vote          ->  contract counts the votes
contract finalises       ->  outcome goes on the ledger
```

You will write two rules by hand, then operate the full contract.

In [ ]:
# Setup: run this cell first.
import os, sys
import numpy, pandas, matplotlib          # loads the packages in JupyterLite
try:
    import sklearn
except ImportError:
    print("scikit-learn not found: agents will use the built-in rule tree instead.")

if not os.path.isdir("wonders"):
    if os.path.isdir("../wonders"):
        sys.path.insert(0, "..")
    else:
        try:                                # Google Colab: upload the workshop zip
            from google.colab import files
            import zipfile
            print("Upload blockchain_wonders.zip")
            name = next(iter(files.upload()))
            zipfile.ZipFile(name).extractall(".")
            sys.path.insert(0, "blockchain_wonders")
        except ImportError:
            print("Cannot find the 'wonders' folder. Keep this notebook next to it.")

import wonders
print("Ready. wonders version", wonders.__version__)

## 1. Write a rule: authorised participation

**Your turn.** Finish `may_submit`: an agent may submit only if its address is in the registry **and** its role there is `"agent"`.

In [ ]:
from wonders import Wallet

registry = {}                                    # address -> role

def register(wallet, role):
    registry[wallet.address] = role

def may_submit(wallet):
    # TODO: return True only when registry has this wallet.address with role "agent"
    return True

agent_a, stranger, validator = Wallet("agent-A"), Wallet("stranger"), Wallet("validator-1")
register(agent_a, "agent")
register(validator, "validator")

for w in (agent_a, stranger, validator):
    print(f"{w.name:<12} may submit? {may_submit(w)}")

## 2. Write a rule: voting

Two numbers define a vote:

- **quorum**: the share of validators who must take part
- **approval threshold**: the share of votes cast that must be YES

**Your turn.** Finish `tally`.

In [ ]:
def tally(votes, n_validators, quorum, threshold):
    """votes is a list of True/False. Returns 'approved', 'rejected' or 'no quorum'."""
    turnout = len(votes) / n_validators
    # TODO: if turnout is below quorum, return "no quorum"
    pass
    yes_share = sum(votes) / len(votes)
    # TODO: return "approved" if yes_share reaches the threshold, otherwise "rejected"
    return "approved"

print(tally([True, True, True, False], n_validators=5, quorum=0.6, threshold=0.6))   # approved
print(tally([True],                    n_validators=5, quorum=0.6, threshold=0.6))   # no quorum
print(tally([True, False, False],      n_validators=5, quorum=0.6, threshold=0.6))   # rejected

## 3. The full contract

`World` sets up everything from Activities 1 and 2: a ledger, three registered agents, five validators, an administrator, and a governance contract. The contract's behaviour is controlled by a **rules dictionary**.

In [ ]:
from wonders import World, WEAK_RULES, RULE_HELP, sample_application

for rule, value in WEAK_RULES.items():
    print(f"{rule:<22} = {str(value):<12} {RULE_HELP[rule]}")

These default rules are deliberately weak. Start with a sensible set and run one case through.

In [ ]:
rules = {
    "require_registration": True,
    "check_model_hash": True,
    "min_agents": 2,
    "quorum": 0.6,
    "approval_threshold": 0.6,
    "one_vote_each": True,
}
world = World(rules)
contract = world.contract

app = sample_application("APP-100")
case_id = world.submit(app)                       # all three agents decide and submit
print("agents said:", [d["decision"] for d in contract.cases[case_id]["decisions"]])

print("before any votes:", contract.finalize(case_id)["status"])

**Your turn.** Have validators vote, then finalise. Use `contract.vote(validator_wallet, case_id, True_or_False, reason)`. The validators are in `world.validators`.

In [ ]:
# TODO: make at least three validators vote, then call contract.finalize(case_id)
result = contract.finalize(case_id)
print(result["status"], "|", result["outcome"], "|", result["why"])

Everything the contract did is on the ledger.

In [ ]:
import pandas as pd
pd.DataFrame(contract.history(case_id))[["block", "kind", "by", "time"]]

## 4. The contract says no

An unregistered agent tries to submit. The contract refuses **and records the attempt**.

In [ ]:
from wonders import DeterministicAgent, GovernanceError

outsider = DeterministicAgent(Wallet("outsider-bot", role="agent"), seed=5)
try:
    contract.submit_decision(outsider.wallet, outsider.decide(sample_application("APP-101")))
except GovernanceError as err:
    print("refused:", err)

print("violations on the ledger:", contract.violations())

**Your turn.** A validator tries to act as an agent. Submit a decision using `world.validators[0]` as the wallet and catch the error. What did the contract check?

In [ ]:
real = world.agents[0].decide(sample_application("APP-102"))
try:
    # TODO: call contract.submit_decision with world.validators[0] and real
    pass
except GovernanceError as err:
    print("refused:", err)

## 5. Rules for human judgement

Two more rules bring people in where machines should not decide alone:

- `human_review_above`: large amounts need a human sign-off
- `on_conflict = "escalate"`: when agents disagree, a human decides

In [ ]:
world = World({**rules, "human_review_above": 200, "on_conflict": "escalate"})
contract = world.contract

big = sample_application("APP-200", amount_requested=400)
world.submit(big)
world.votes("APP-200", yes=5)
r = contract.finalize("APP-200")
print(r["status"], "|", r["why"])

**Your turn.** You are the fund administrator (`world.admin`). Sign the case off with `contract.human_signoff(admin, case_id, "APPROVE" or "REJECT", note)` and finalise again.

In [ ]:
# TODO: call contract.human_signoff(...) then contract.finalize("APP-200")
r = contract.finalize("APP-200")
print(r["status"], "|", r["outcome"], "|", r["why"])
pd.DataFrame(contract.history("APP-200"))[["block", "kind", "by"]]

## 6. Design exercise

The fund's board has written this policy:

> *"Every application is assessed by all three agents. If they disagree, a person decides. Any request above 150 thousand rupees needs a person. A decision stands only if most validators took part and at least two-thirds of those voting agreed. Nobody outside the fund takes part."*

**Your turn.** Translate the policy into a rules dictionary.

In [ ]:
board_rules = {
    # TODO: fill in the rules that express the board's policy
    "require_registration": False,
}

world = World(board_rules)
world.submit(sample_application("APP-300"))
world.votes("APP-300", yes=4, no=1)
r = world.contract.finalize("APP-300")
print("small request :", r["status"], r["outcome"])

world.submit(sample_application("APP-301", amount_requested=300))
world.votes("APP-301", yes=5)
print("large request :", world.contract.finalize("APP-301")["status"])

Expected: the small request is `final APPROVE`; the large one is `needs_human`.

## Discuss with your neighbour (3 minutes)

1. The rules dictionary is itself written to the ledger when the contract is deployed. Why does that matter?
2. Who should be allowed to change the rules, and how?
3. Which of today's rules would you never automate?

**Next:** Activity 4. Things go wrong.